# クイック提出 10 本（ベース = 現時点最高）

**ベース**: **movie_title_info（small）+ PCA 16** → **Public 0.75449**（現時点最高）。  
この設定を土台に、**次元削減は PCA のみ**で、ドキュメントでまだ試せてない案を **10 本** 提出用 CSV で作成する。

| # | 提出ファイル | 内容 |
|---|-------------|------|
| 1 | seed_avg | シード 42,43,44 で 3 本学習 → 予測を平均 |
| 2 | pca24 | PCA **24** 次元で 1 本 |
| 3 | large_pca16 | movie_title_info_large + PCA 16（単体 1 本） |
| 4 | ensemble_two | ベース + large_pca16 を 0.5:0.5 で平均 |
| 5 | publisher_freq | ベース + publisher_name 出現回数 1 列（01_PREPROCESS） |
| 6 | lda_10 | ベース + movie_info の LDA 10 次元（01/03） |
| 7 | doc_x_critic_te | ベース + critic_te × genre_Documentary 1 列（02 ベスト単体） |
| 8 | pca32 | PCA **32** 次元で 1 本 |
| 9 | pca8 | PCA **8** 次元で 1 本（次元を下げた版） |
| 10 | blend_pca16_pca24 | pca16 と pca24 の予測を 0.5:0.5 でブレンド |

先頭で **ベース pca16**（0.75449 と同じ 1 本）も作成するので、**計 11 本**。提出回数 10 回なら、ベースを除く 10 本や、好きな 10 本を選んで提出すればよい。

## 1. インポート・設定・データ読み込み

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")

from lib import get_baseline_data, BASELINE_FEATURES, BASELINE_LGB_PARAMS
from lib.embedding_reduction import fit_transform_embedding
from lib.submission import save_submission, verify_submission
from lib.text_vectors import build_vectors

OUTPUT_DIR = Path("outputs")
EMBEDDINGS_DIR = OUTPUT_DIR / "embeddings"
SUBMISSIONS_DIR = OUTPUT_DIR / "submissions"
os.makedirs(SUBMISSIONS_DIR, exist_ok=True)

EMBEDDING_NAME = "movie_title_info"  # 現時点最高 0.75449 のベース
N_COMPONENTS = 16
EMBEDDING_CONFIGS = {
    "movie_info": {"path": EMBEDDINGS_DIR / "movie_info_embeddings.pkl", "loader": "movie_info"},
    "movie_info_large": {"path": EMBEDDINGS_DIR / "movie_info_embeddings_large.pkl", "loader": "movie_info"},
    "movie_title_info": {"path": EMBEDDINGS_DIR / "movie_title_info_embeddings.pkl", "loader": "title_info"},
    "movie_title_info_large": {"path": EMBEDDINGS_DIR / "movie_title_info_embeddings_large.pkl", "loader": "title_info"},
}

def load_embeddings(name):
    config = EMBEDDING_CONFIGS[name]
    path = Path(config["path"])
    if not path.exists():
        path = path.with_suffix(".parquet")
    if not path.exists():
        raise FileNotFoundError(f"embedding がありません: {config['path']}")
    if config["loader"] == "movie_info":
        from lib.openai_embeddings import load_movie_info_embeddings
        return load_movie_info_embeddings(path=path)
    from lib.openai_embeddings import load_movie_title_info_embeddings
    return load_movie_title_info_embeddings(path=path)

def _merge_embeddings(df, emb_df):
    emb_cols = [c for c in emb_df.columns if c != "rotten_tomatoes_link"]
    merged = df[["rotten_tomatoes_link"]].merge(
        emb_df[["rotten_tomatoes_link"] + emb_cols], on="rotten_tomatoes_link", how="left"
    )
    return merged[emb_cols].fillna(0).astype(np.float32).values

train, test = get_baseline_data()
y = train["target"].values
base_features = list(BASELINE_FEATURES)
safe = EMBEDDING_NAME.replace(" ", "_")

emb_df = load_embeddings(EMBEDDING_NAME)
E_train = _merge_embeddings(train, emb_df)
E_test = _merge_embeddings(test, emb_df)

print(f"ベース: {EMBEDDING_NAME} + PCA {N_COMPONENTS} (Public 0.75449)")
print(f"Train: {len(train):,}, Test: {len(test):,}, Embedding: {E_train.shape[1]} 次元")
print(f"提出先: {SUBMISSIONS_DIR}")

## 2. 11 本の提出ファイルを作成

In [ ]:
assert "ID" in test.columns, "test に ID 列がありません"
n_test = len(test)
saved = []

def run_pca_save(E_tr, E_te, n_comp, seed, fname, tag):
    train_r, test_r, prefix = fit_transform_embedding(E_tr, E_te, method="pca", n_components=n_comp, random_state=seed)
    red = [f"{prefix}_{i}" for i in range(train_r.shape[1])]
    feats = base_features + red
    X_tr = pd.concat([train[base_features].reset_index(drop=True), pd.DataFrame(train_r, columns=red)], axis=1)
    X_te = pd.concat([test[base_features].reset_index(drop=True), pd.DataFrame(test_r, columns=red)], axis=1)
    model = lgb.LGBMClassifier(**BASELINE_LGB_PARAMS)
    model.fit(X_tr[feats], y)
    pred = model.predict_proba(X_te[feats])[:, 1]
    assert len(pred) == n_test, f"予測長 {len(pred)} != test 行数 {n_test}"
    path = SUBMISSIONS_DIR / fname
    save_submission(test, pred, path, sanitize=True)
    v = verify_submission(path, test)
    print(f"  [{tag}] → {fname}  ({'OK' if v['ok'] else v['message']})")
    saved.append(fname)

# 0. ベース pca16（0.75449 の 1 本・ensemble_two 用にも必要）
run_pca_save(E_train, E_test, N_COMPONENTS, 42, f"submission_embedding_{safe}_pca{N_COMPONENTS}.csv", "pca16")

# 1. seed_avg
preds = []
for rs in [42, 43, 44]:
    train_r, test_r, prefix = fit_transform_embedding(E_train, E_test, method="pca", n_components=N_COMPONENTS, random_state=rs)
    red = [f"{prefix}_{i}" for i in range(train_r.shape[1])]
    feats = base_features + red
    X_tr = pd.concat([train[base_features].reset_index(drop=True), pd.DataFrame(train_r, columns=red)], axis=1)
    X_te = pd.concat([test[base_features].reset_index(drop=True), pd.DataFrame(test_r, columns=red)], axis=1)
    model = lgb.LGBMClassifier(**BASELINE_LGB_PARAMS)
    model.fit(X_tr[feats], y)
    preds.append(model.predict_proba(X_te[feats])[:, 1])
fname = f"submission_embedding_{safe}_pca{N_COMPONENTS}_seed_avg.csv"
save_submission(test, np.mean(preds, axis=0), SUBMISSIONS_DIR / fname, sanitize=True)
v = verify_submission(SUBMISSIONS_DIR / fname, test)
print(f"  [seed_avg] → {fname}  ({'OK' if v['ok'] else v['message']})")
saved.append(fname)

# 2. pca24
run_pca_save(E_train, E_test, 24, 42, f"submission_embedding_{safe}_pca24.csv", "pca24")

# 3. large_pca16（単体）
try:
    emb_large = load_embeddings("movie_title_info_large")
    E_tr_l = _merge_embeddings(train, emb_large)
    E_te_l = _merge_embeddings(test, emb_large)
    run_pca_save(E_tr_l, E_te_l, N_COMPONENTS, 42, "submission_embedding_movie_title_info_large_pca16.csv", "large_pca16")
except FileNotFoundError as e:
    print(f"  [large_pca16] スキップ: {e}")

# 4. ensemble_two（pca16 + large_pca16 の 0.5:0.5）
p1 = SUBMISSIONS_DIR / f"submission_embedding_{safe}_pca{N_COMPONENTS}.csv"
p2 = SUBMISSIONS_DIR / "submission_embedding_movie_title_info_large_pca16.csv"
if p1.exists() and p2.exists():
    d1, d2 = pd.read_csv(p1), pd.read_csv(p2)
    merged = d1[["ID", "target"]].merge(d2[["ID", "target"]], on="ID", suffixes=("_1", "_2"))
    merged["target"] = (merged["target_1"].astype(float) + merged["target_2"].astype(float)) / 2
    test_p = test[["ID"]].merge(merged[["ID", "target"]], on="ID", how="left")
    if not test_p["target"].isna().any():
        save_submission(test, test_p["target"].values, SUBMISSIONS_DIR / "submission_embedding_ensemble_movie_title_info_and_large_pca16.csv", sanitize=True)
        v = verify_submission(SUBMISSIONS_DIR / "submission_embedding_ensemble_movie_title_info_and_large_pca16.csv", test)
        print(f"  [ensemble_two] → submission_embedding_ensemble_movie_title_info_and_large_pca16.csv  ({'OK' if v['ok'] else v['message']})")
        saved.append("submission_embedding_ensemble_movie_title_info_and_large_pca16.csv")
else:
    print(f"  [ensemble_two] スキップ: {p1.name} または {p2.name} がありません（§3 で large_pca16 を作成後に再実行）")

# 5. publisher_freq
if "publisher_name" in train.columns:
    pub_count = train["publisher_name"].value_counts()
    train_pub = train.copy()
    test_pub = test.copy()
    train_pub["publisher_name_freq"] = train_pub["publisher_name"].map(pub_count).fillna(0).astype(np.int32)
    test_pub["publisher_name_freq"] = test_pub["publisher_name"].map(pub_count).fillna(0).astype(np.int32)
    base_freq = base_features + ["publisher_name_freq"]
    train_r, test_r, prefix = fit_transform_embedding(E_train, E_test, method="pca", n_components=N_COMPONENTS, random_state=42)
    red = [f"{prefix}_{i}" for i in range(train_r.shape[1])]
    feats = base_freq + red
    X_tr = pd.concat([train_pub[base_freq].reset_index(drop=True), pd.DataFrame(train_r, columns=red)], axis=1)
    X_te = pd.concat([test_pub[base_freq].reset_index(drop=True), pd.DataFrame(test_r, columns=red)], axis=1)
    model = lgb.LGBMClassifier(**BASELINE_LGB_PARAMS)
    model.fit(X_tr[feats], y)
    pred = model.predict_proba(X_te[feats])[:, 1]
    fname = f"submission_embedding_{safe}_pca{N_COMPONENTS}_publisher_freq.csv"
    save_submission(test, pred, SUBMISSIONS_DIR / fname, sanitize=True)
    v = verify_submission(SUBMISSIONS_DIR / fname, test)
    print(f"  [publisher_freq] → {fname}  ({'OK' if v['ok'] else v['message']})")
    saved.append(fname)
else:
    print("  [publisher_freq] スキップ: publisher_name なし")

# 6. lda_10（movie_info の LDA 10 次元を追加）
mi_tr = train["movie_info"].astype(str)
mi_te = test["movie_info"].astype(str)
tr_mat, _, te_mat, lda_prefix = build_vectors("lda_10", mi_tr, None, mi_te)
if tr_mat is not None and te_mat is not None:
    lda_cols = [f"{lda_prefix}_{i}" for i in range(tr_mat.shape[1])]
    train_lda = train.copy()
    test_lda = test.copy()
    for i, c in enumerate(lda_cols):
        train_lda[c] = tr_mat[:, i]
        test_lda[c] = te_mat[:, i]
    base_lda = base_features + lda_cols
    train_r, test_r, prefix = fit_transform_embedding(E_train, E_test, method="pca", n_components=N_COMPONENTS, random_state=42)
    red = [f"{prefix}_{i}" for i in range(train_r.shape[1])]
    feats = base_lda + red
    X_tr = pd.concat([train_lda[base_lda].reset_index(drop=True), pd.DataFrame(train_r, columns=red)], axis=1)
    X_te = pd.concat([test_lda[base_lda].reset_index(drop=True), pd.DataFrame(test_r, columns=red)], axis=1)
    model = lgb.LGBMClassifier(**BASELINE_LGB_PARAMS)
    model.fit(X_tr[feats], y)
    pred = model.predict_proba(X_te[feats])[:, 1]
    fname = f"submission_embedding_{safe}_pca{N_COMPONENTS}_lda10.csv"
    save_submission(test, pred, SUBMISSIONS_DIR / fname, sanitize=True)
    v = verify_submission(SUBMISSIONS_DIR / fname, test)
    print(f"  [lda_10] → {fname}  ({'OK' if v['ok'] else v['message']})")
    saved.append(fname)
else:
    print("  [lda_10] スキップ: build_vectors 失敗")

# 7. doc_x_critic_te（critic_te × genre_Documentary）
train_doc = train.copy()
test_doc = test.copy()
train_doc["critic_te_x_genre_Documentary"] = train_doc["critic_name_te_ts"].astype(float) * train_doc["genre_Documentary"].astype(float)
test_doc["critic_te_x_genre_Documentary"] = test_doc["critic_name_te_ts"].astype(float) * test_doc["genre_Documentary"].astype(float)
base_doc = base_features + ["critic_te_x_genre_Documentary"]
train_r, test_r, prefix = fit_transform_embedding(E_train, E_test, method="pca", n_components=N_COMPONENTS, random_state=42)
red = [f"{prefix}_{i}" for i in range(train_r.shape[1])]
feats = base_doc + red
X_tr = pd.concat([train_doc[base_doc].reset_index(drop=True), pd.DataFrame(train_r, columns=red)], axis=1)
X_te = pd.concat([test_doc[base_doc].reset_index(drop=True), pd.DataFrame(test_r, columns=red)], axis=1)
model = lgb.LGBMClassifier(**BASELINE_LGB_PARAMS)
model.fit(X_tr[feats], y)
pred = model.predict_proba(X_te[feats])[:, 1]
fname = f"submission_embedding_{safe}_pca{N_COMPONENTS}_doc_x_critic_te.csv"
save_submission(test, pred, SUBMISSIONS_DIR / fname, sanitize=True)
v = verify_submission(SUBMISSIONS_DIR / fname, test)
print(f"  [doc_x_critic_te] → {fname}  ({'OK' if v['ok'] else v['message']})")
saved.append(fname)

# 8. pca32
run_pca_save(E_train, E_test, 32, 42, f"submission_embedding_{safe}_pca32.csv", "pca32")

# 9. pca8（次元を下げた版）
run_pca_save(E_train, E_test, 8, 42, f"submission_embedding_{safe}_pca8.csv", "pca8")

# 10. blend_pca16_pca24（pca16 と pca24 の予測を 0.5:0.5 でブレンド）
path_16 = SUBMISSIONS_DIR / f"submission_embedding_{safe}_pca{N_COMPONENTS}.csv"
path_24 = SUBMISSIONS_DIR / f"submission_embedding_{safe}_pca24.csv"
if path_16.exists() and path_24.exists():
    d16 = pd.read_csv(path_16)
    d24 = pd.read_csv(path_24)
    merged = d16[["ID", "target"]].merge(d24[["ID", "target"]], on="ID", suffixes=("_16", "_24"))
    merged["target"] = (merged["target_16"].astype(float) + merged["target_24"].astype(float)) / 2
    test_p = test[["ID"]].merge(merged[["ID", "target"]], on="ID", how="left")
    if not test_p["target"].isna().any():
        fname_blend = f"submission_embedding_{safe}_blend_pca16_pca24.csv"
        save_submission(test, test_p["target"].values, SUBMISSIONS_DIR / fname_blend, sanitize=True)
        v = verify_submission(SUBMISSIONS_DIR / fname_blend, test)
        print(f"  [blend_pca16_pca24] → {fname_blend}  ({'OK' if v['ok'] else v['message']})")
        saved.append(fname_blend)
else:
    print("  [blend_pca16_pca24] スキップ: pca16 または pca24 がありません")

print(f"\n保存先: {SUBMISSIONS_DIR}")
print(f"作成: {len(saved)} 件")
print("\n--- 提出ファイル検証 ---")
for fname in saved:
    p = SUBMISSIONS_DIR / fname
    if p.exists():
        df = pd.read_csv(p)
        ok = len(df) == n_test and list(df.columns) == ["ID", "target"] and df["target"].between(0, 1).all()
        status = "OK" if ok else f"NG (行数={len(df)}, target範囲=[{df['target'].min():.3f}, {df['target'].max():.3f}])"
        print(f"  {fname}: {status}")